# Colab Synthetic Markdown Index (public-safe PoC)

> **Public synthetic data only.** This notebook does **not** read private repositories, ObsidianBook, local files, Google Drive, credentials or GitHub Actions. Do not paste personal notes into cells. It is **not** a CI runner or a background job.

1. Open from the public GitHub branch in Google Colab, choose **CPU** runtime first, and run cells manually top to bottom.
2. The next cell installs pinned, publicly available packages; it needs internet access to PyPI and the public Hugging Face model.
3. Optional GPU: manually change `USE_GPU = False` to `True`, only if your Colab account allows a GPU.
4. Review the **public-only summary**; a separate cell permits an optional local JSON export. Actual Compute Units must be read from your own Colab UI, never inferred here.

Official limits: https://research.google.com/colaboratory/intl/zh-TW/faq.html


In [ ]:
# Install packages into this interactive Colab runtime (no API keys).
%pip install -q "sentence-transformers==5.1.2" "huggingface-hub==0.36.0"


In [ ]:
# UNIT:CORE
import json
import math
import re
import time

MARKER = "SYNTHETIC_PUBLIC_DEMO"
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
TOPICS = (
    ("graph", "nodes edges graph relationships knowledge map"),
    ("testing", "unit test failure validation assertion coverage"),
    ("product", "product roadmap customer discovery requirement prioritization"),
    ("design", "interface typography layout accessibility user experience"),
    ("automation", "scheduled tasks retry queue workflow orchestration"),
    ("analytics", "data metrics trends measurement visualization"),
)
SCHEMA_KEYS = frozenset((
    "schema_version", "experiment", "dataset_kind", "status", "document_count",
    "chunk_count", "embedding_dimension", "model_id", "model_revision",
    "runtime_mode", "gpu_requested", "gpu_fallback", "elapsed_seconds",
    "compute_units_observed"
))

class UnsafeSyntheticInput(ValueError):
    pass


def make_synthetic_documents(count=120):
    if type(count) is not int or not 100 <= count <= 500:
        raise UnsafeSyntheticInput("Expected 100..500 generated synthetic documents")
    docs = []
    for i in range(count):
        topic, keywords = TOPICS[i % len(TOPICS)]
        body = (
            f"{MARKER}\n# Synthetic {topic} note {i:03d}\n\n"
            f"Topic: {topic}. Vocabulary: {keywords}. This sample was generated "
            f"for a demonstration and contains no actual personal knowledge. "
            f"Linked sample: [synthetic demo](synthetic://demo-{(i + 1) % count:03d}). "
            f"Review and improve {topic} experiments using synthetic reference records."
        )
        if i != 0 and i % 19 == 0:
            # Synthetic near-duplicates for basic indexing / duplicate experiments.
            body += " Synthetic repeated sentence. Synthetic repeated sentence."
        docs.append({"id": f"demo-{i:03d}", "title": f"Synthetic {topic} note {i:03d}",
                     "topic": topic, "markdown": body})
    validate_synthetic_documents(docs)
    return docs


def validate_synthetic_documents(docs):
    if type(docs) is not list or not 100 <= len(docs) <= 500:
        raise UnsafeSyntheticInput("Synthetic dataset must contain 100..500 documents")
    seen = set()
    allowed_topics = {item[0] for item in TOPICS}
    for row in docs:
        if type(row) is not dict or set(row) != {"id", "title", "topic", "markdown"}:
            raise UnsafeSyntheticInput("Unexpected synthetic document fields")
        if any(type(row[k]) is not str for k in row):
            raise UnsafeSyntheticInput("Synthetic document fields must be strings")
        if not re.fullmatch(r"demo-[0-9]{3}", row["id"]) or row["id"] in seen:
            raise UnsafeSyntheticInput("Invalid or duplicate synthetic identifier")
        seen.add(row["id"])
        if row["topic"] not in allowed_topics or not row["title"].startswith("Synthetic "):
            raise UnsafeSyntheticInput("Unexpected synthetic topic or title")
        if not row["markdown"].startswith(MARKER + "\n") or len(row["markdown"]) > 2000:
            raise UnsafeSyntheticInput("Unmarked or oversized synthetic document")
    return True


def build_public_report(*, count, dimension, revision, runtime, gpu_requested,
                        elapsed_seconds, status="PASS"):
    report = {
        "schema_version": 1,
        "experiment": "colab-synthetic-poc",
        "dataset_kind": "synthetic",
        "status": status,
        "document_count": count,
        "chunk_count": count,  # One short synthetic document is one chunk in this PoC.
        "embedding_dimension": dimension,
        "model_id": MODEL_ID,
        "model_revision": revision,
        "runtime_mode": runtime,
        "gpu_requested": gpu_requested,
        "gpu_fallback": bool(gpu_requested and runtime == "CPU"),
        "elapsed_seconds": elapsed_seconds,
        "compute_units_observed": None,  # Must be read manually from the Colab UI.
    }
    validate_public_report(report)
    return report


def validate_public_report(report):
    if type(report) is not dict or set(report) != SCHEMA_KEYS:
        raise ValueError("Invalid public report fields")
    if report["schema_version"] != 1 or type(report["schema_version"]) is not int:
        raise ValueError("Invalid schema version")
    if report["experiment"] != "colab-synthetic-poc" or report["dataset_kind"] != "synthetic":
        raise ValueError("Invalid experiment classification")
    if report["status"] not in ("PASS", "FAIL", "UNVERIFIED", "INTERRUPTED"):
        raise ValueError("Invalid experiment status")
    if any(type(report[k]) is not int or not 100 <= report[k] <= 500 for k in ("document_count", "chunk_count")):
        raise ValueError("Invalid document / chunk counts")
    if report["chunk_count"] != report["document_count"]:
        raise ValueError("Unexpected PoC chunking")
    if type(report["embedding_dimension"]) is not int or not 1 <= report["embedding_dimension"] <= 8192:
        raise ValueError("Invalid embedding dimension")
    if report["model_id"] != MODEL_ID or type(report["model_revision"]) is not str or not re.fullmatch(r"[0-9a-f]{40}", report["model_revision"]):
        raise ValueError("Invalid public model / revision")
    if report["runtime_mode"] not in ("CPU", "GPU"):
        raise ValueError("Invalid runtime mode")
    if any(type(report[k]) is not bool for k in ("gpu_requested", "gpu_fallback")):
        raise ValueError("Invalid GPU flags")
    if report["gpu_fallback"] != (report["gpu_requested"] and report["runtime_mode"] == "CPU"):
        raise ValueError("Inconsistent GPU fallback")
    if type(report["elapsed_seconds"]) not in (int, float) or not math.isfinite(report["elapsed_seconds"]) or report["elapsed_seconds"] < 0:
        raise ValueError("Invalid elapsed time")
    if report["compute_units_observed"] is not None:
        raise ValueError("Compute Units can only be recorded manually, outside the public report")
    return True


In [ ]:
# Generate and validate 120 public, synthetic Markdown documents in memory.
start_time = time.perf_counter()
documents = make_synthetic_documents(120)
print(f"Synthetic documents validated: {len(documents)}")


In [ ]:
# OPTIONAL GPU. This PoC uses CPU unless the user explicitly requests GPU.
USE_GPU = False

try:
    import torch
    from huggingface_hub import model_info
    from sentence_transformers import SentenceTransformer

    gpu_available = bool(torch.cuda.is_available())
    device = "cuda" if USE_GPU and gpu_available else "cpu"
    if USE_GPU and not gpu_available:
        print("GPU not available; running on CPU instead")
    # Resolve and then pin the public model revision for this execution.
    model_revision = model_info(MODEL_ID, token=False).sha
    if not isinstance(model_revision, str) or not re.fullmatch(r"[0-9a-f]{40}", model_revision):
        raise RuntimeError("Invalid public model revision")
    model = SentenceTransformer(MODEL_ID, revision=model_revision, device=device, token=False)
except Exception:
    raise RuntimeError("Public model initialization failed; no PASS report was generated") from None


In [ ]:
# Encode only validated synthetic Markdown; never read files or other sources.
try:
    import numpy as np

    validate_synthetic_documents(documents)
    snippets = [doc["markdown"] for doc in documents]
    vectors = np.asarray(model.encode(snippets, normalize_embeddings=True, show_progress_bar=False))
    query = "synthetic graph nodes edges relationships"
    query_vector = np.asarray(model.encode([query], normalize_embeddings=True, show_progress_bar=False))[0]
    if vectors.ndim != 2 or vectors.shape[0] != len(documents) or vectors.shape[1] <= 0:
        raise RuntimeError("Unexpected vector shape")
    if not np.isfinite(vectors).all() or not np.isfinite(query_vector).all():
        raise RuntimeError("Non-finite embeddings")
    scores = vectors @ query_vector
    top_indices = np.argsort(-scores)[:5].tolist()
    if not top_indices:
        raise RuntimeError("No ranked search results")
    # No document text, query, vectors, search result IDs or content in the public report.
    ranking_count = len(top_indices)
except Exception:
    raise RuntimeError("Synthetic embedding smoke test failed; no PASS report was generated") from None


In [ ]:
# Report metadata only. No content, embeddings, queries, paths or exception traces.
public_report = build_public_report(
    count=len(documents), dimension=int(vectors.shape[1]),
    revision=model_revision, runtime="GPU" if device == "cuda" else "CPU",
    gpu_requested=USE_GPU, elapsed_seconds=round(time.perf_counter() - start_time, 3),
)
print(json.dumps(public_report, ensure_ascii=False, indent=2, allow_nan=False))
print(f"Search smoke test returned {ranking_count} synthetic results (content hidden).")


## Optional manual export

The report contains **only allowlisted synthetic experiment metadata**. Leave the switch off by default. If you explicitly want to inspect/download it, set `SAVE_PUBLIC_REPORT = True` and run the next cell. Do not upload a real vault, an index, a GitHub secret or an unreviewed notebook output to a public issue. The notebook does not record Colab CU automatically.


In [ ]:
SAVE_PUBLIC_REPORT = False
if SAVE_PUBLIC_REPORT:
    validate_public_report(public_report)
    with open("colab-synthetic-public-report.json", "w", encoding="utf-8") as handle:
        json.dump(public_report, handle, ensure_ascii=False, indent=2, allow_nan=False)
    print("Wrote colab-synthetic-public-report.json (manual download via Colab Files pane)")
else:
    print("No report file created; enable manual export explicitly if needed")
